In [24]:
from google import genai
from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv()
client =genai.Client()

interaction = client.interactions.create(
    model= "gemini-3.7-flash",
    input="請逐步教學含指令, 如何使用raspberry樹莓派架設出家用的NAS伺服器?")
display(Markdown(interaction.output_text))
      

使用樹莓派（Raspberry Pi）架設家用 NAS（網路附加儲存）最經典、穩定且效能最好的方式是透過 **Samba (SMB/CIFS)** 協定。

以下為您整理完整的**逐步安裝教學與指令**（建議使用 Raspberry Pi 4 或 5，搭配 USB 3.0 外接硬碟以獲得最佳讀寫速度）。

---

### 前置準備
1. **硬體**：Raspberry Pi（建議 Pi 4B 或 5）、MicroSD 卡、外接硬碟（建議自帶獨立電源）、網路線。
2. **系統**：已安裝 Raspberry Pi OS（建議 64-bit Lite 或 Desktop 版），並已開啟 SSH 連線。

---

### 第一步：系統更新與套件安裝

1. 透過 SSH 登入樹莓派後，先更新系統套件：
   ```bash
   sudo apt update && sudo apt upgrade -y
   ```

2. 安裝 Samba 服務與檔案系統支援工具（支援 NTFS/exFAT）：
   ```bash
   sudo apt install samba samba-common-bin ntfs-3g exfat-fuse -y
   ```

---

### 第二步：外接硬碟格式化與掛載

#### 1. 查詢硬碟代號
接上外接硬碟，執行以下指令確認硬碟分割區名稱：
```bash
lsblk
```
*(通常會看到 `sda`、`sda1` 或 `sdb1`。以下範例假設硬碟分割區為 `/dev/sda1`)*

#### 2. 取得硬碟的 UUID
為了避免重開機後硬碟代號改變，建議使用 UUID 進行掛載：
```bash
sudo blkid
```
找到 `/dev/sda1` 那一行，複製其 `UUID="xxxx-xxxx"` 以及 `TYPE="ext4"`（或 `ntfs`, `exfat`）。

#### 3. 建立掛載目錄
```bash
sudo mkdir -p /mnt/nas_disk
```

#### 4. 設定開機自動掛載 (`/etc/fstab`)
編輯系統掛載設定檔：
```bash
sudo nano /etc/fstab
```
在檔案的最下方加入一行（**請將 UUID 與 TYPE 替換為剛才查詢到的資訊**）：

* 如果硬碟是 **ext4**（Linux 推薦格式，速度最快）：
  ```text
  UUID=你的UUID /mnt/nas_disk ext4 defaults,nofail 0 2
  ```
* 如果硬碟是 **NTFS**（Windows 格式）：
  ```text
  UUID=你的UUID /mnt/nas_disk ntfs-3g defaults,nofail,uid=1000,gid=1000,umask=007 0 0
  ```
*(註：`nofail` 參數可確保硬碟沒接上時，樹莓派依然能正常開機)*

按 `Ctrl + O` 存檔，按 `Enter`，再按 `Ctrl + X` 離開。

#### 5. 測試掛載
執行以下指令測試是否成功掛載，且沒有報錯：
```bash
sudo mount -a
df -h
```
*(在 `df -h` 的結果中，看到 `/mnt/nas_disk` 代表掛載成功)*

---

### 第三步：設定共用資料夾與權限

1. 在外接硬碟中建立一個共用資料夾：
   ```bash
   sudo mkdir -p /mnt/nas_disk/shares
   ```

2. 變更資料夾權限（以預設使用者 `pi` 為例，若你的使用者名稱不同請自行修改）：
   ```bash
   sudo chown -R $USER:$USER /mnt/nas_disk/shares
   sudo chmod -R 775 /mnt/nas_disk/shares
   ```

---

### 第四步：設定 Samba (SMB) 伺服器

1. 備份原始設定檔：
   ```bash
   sudo cp /etc/samba/smb.conf /etc/samba/smb.conf.bak
   ```

2. 編輯 Samba 設定檔：
   ```bash
   sudo nano /etc/samba/smb.conf
   ```

3. 移動到檔案**最底部**，貼上以下共用設定：
   ```ini
   [HomeNAS]
      comment = Raspberry Pi NAS
      path = /mnt/nas_disk/shares
      browseable = yes
      writeable = yes
      create mask = 0775
      directory mask = 0775
      public = no
      valid users = @samba_users
   ```
   按 `Ctrl + O` 存檔，`Enter`，`Ctrl + X` 離開。

4. 建立 Samba 使用者群組與密碼：
   ```bash
   # 建立使用者群組
   sudo groupadd samba_users
   
   # 將目前使用者加入群組
   sudo usermod -aG samba_users $USER
   
   # 為目前使用者設定 Samba 登入密碼（會提示輸入兩次密碼，此為日後連線密碼）
   sudo smbpasswd -a $USER
   ```

5. 重啟 Samba 服務以套用設定：
   ```bash
   sudo systemctl restart smbd
   sudo systemctl restart nmbd
   ```

---

### 第五步：取得樹莓派 IP 並進行連線測試

1. 查詢樹莓派的區域網路 IP：
   ```bash
   hostname -I
   ```
   *(例如：`192.168.1.100`)*

#### 【Windows 電腦連線】
1. 開啟「檔案總管」，在上方網址列輸入：
   `\\192.168.1.100\HomeNAS`
2. 輸入剛才設定的**使用者帳號**與 **Samba 密碼**。
3. 可在「本機」右鍵選擇「連線網路磁碟機」，將其固定為一個磁碟槽（如 Z: 槽）。

#### 【Mac 電腦連線】
1. 開啟 Finder，按快捷鍵 `Command + K`。
2. 伺服器位址輸入：
   `smb://192.168.1.100/HomeNAS`
3. 點擊「連線」並輸入帳號與 Samba 密碼。

#### 【iOS / Android 手機連線】
* **iOS**：開啟內建「檔案」App -> 點右上角 `...` ->「連接伺服器」-> 輸入 `smb://192.168.1.100`。
* **Android**：下載支援 SMB 的檔案總管（如 *Solid Explorer* 或 *CX 檔案總管*）-> 新增 LAN/SMB 儲存空間連線。

---

### 實用維護與優化建議 (Tips)
1. **設定固定 IP (Static IP)**：建議至家中的 Wi-Fi 路由器後台，將樹莓派的 MAC 地址綁定為固定 IP，避免重開機後 IP 跑掉導致連線中斷。
2. **硬碟休眠 (省電與保護硬碟)**：
   若長時間不讀寫硬碟，可安裝 `hdparm` 來控制硬碟自動休眠：
   ```bash
   sudo apt install hdparm -y
   # 設定 10 分鐘 (120 * 5秒) 無動作即休眠
   sudo hdparm -S 120 /dev/sda
   ```
3. **供電問題**：請勿直接用樹莓派的 USB 孔同時帶動多顆 2.5 吋未外接電源的硬碟，否則可能因電力不足導致斷線或損壞硬碟，建議使用有獨立供電的 USB 集線器 (Powered USB Hub)。